# 💾 Manejo de Archivos — del CRUD en memoria al CRUD persistente
## Programación I — UADE

---

Hasta ahora el CRUD de usuarios guarda los datos **en memoria**: la lista `usuarios` vive en `src/datos.py` y cada vez que cerramos el programa, los usuarios que creamos se pierden.

En este notebook vamos a aprender a **persistir** esos datos en archivos, de manera que sobrevivan entre ejecuciones. Al final vamos a reemplazar `datos.py` por un archivo `data/usuarios.json` y un módulo nuevo: `storage.py`.

### Hoja de ruta

| Bloque | Tema |
|---|---|
| 1 | Rutas: cómo Python encuentra los archivos |
| 2 | `open()`, modos y `with` — las bases |
| 3 | TXT — lectura y escritura simple |
| 4 | CSV — para listas de listas |
| 5 | JSON — para listas de diccionarios ⭐ |
| 6 | `read_data()` y `write_data()` — el puente con el CRUD de usuarios |

> ⭐ El formato más relevante para el Proyecto Final: JSON mapea directamente a la lista de diccionarios `usuarios` que ya usamos.

---
## Bloque 1 — Rutas: cómo Python encuentra los archivos

Antes de abrir cualquier archivo hay que saber **dónde está parado Python** cuando ejecuta el script.

### Ruta absoluta vs ruta relativa

| Tipo | Descripción | Ejemplo |
|---|---|---|
| **Absoluta** | La ruta completa desde la raíz del sistema de archivos | `C:/Users/ana/proyecto/data/usuarios.json` |
| **Relativa** | Relativa al directorio de trabajo actual (`cwd`) | `data/usuarios.json` |

La ruta relativa es **preferible** porque el proyecto funciona en cualquier máquina sin modificar rutas.

### Directorio de trabajo actual (cwd)

Python siempre tiene un "directorio actual" desde donde resuelve las rutas relativas.
Es el directorio desde donde ejecutamos el script, **no** donde está el script.

In [ ]:
import os

# Ver el directorio de trabajo actual
print("Directorio actual (cwd):", os.getcwd())

# Construir rutas de forma segura con os.path.join
# Esto funciona en Windows, Mac y Linux sin cambios
ruta = os.path.join("data", "usuarios.json")
print("Ruta construida:", ruta)

# Verificar si un archivo o carpeta existe
print("¿Existe la carpeta 'data'?", os.path.exists("data"))

### Estructura del proyecto de usuarios

Así queda el proyecto después de esta clase (`datos.py` desaparece y aparecen `storage.py` y `data/`):

```
proyecto/
├── main.py              ← se ejecuta desde acá: python main.py
├── src/
│   ├── interface.py     ← input() / print()
│   ├── services.py      ← lógica del CRUD
│   ├── storage.py       ← NUEVO: read_data() / write_data()
│   └── utils.py         ← validaciones
└── data/
    └── usuarios.json    ← NUEVO: reemplaza a datos.py
```

> ⚠️ Si ejecutás `python main.py` desde **otra** carpeta (por ejemplo desde `src/`), una ruta relativa como `"data/usuarios.json"` se resuelve contra ese cwd y **no va a encontrar** la carpeta `data/`.

### La solución: rutas relativas al archivo, no al cwd

En `services.py` conviene construir la ruta a partir de la ubicación del propio archivo con `pathlib`.
`__file__` es la ruta del `.py` que se está ejecutando, por eso este fragmento **no corre en un notebook** — va en `src/services.py`:

```python
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent              # .../proyecto/src
RUTA_USUARIOS = BASE_DIR.parent / "data" / "usuarios.json"  # .../proyecto/data/usuarios.json
```

Así el programa encuentra el JSON sin importar desde dónde se lo ejecute.

In [ ]:
import os

# Crear la carpeta data/ si no existe (igual que "CREATE TABLE IF NOT EXISTS")
os.makedirs("data", exist_ok=True)
print("Carpeta 'data/' lista.")

# exist_ok=True evita el error si la carpeta ya existe

---
## Bloque 2 — `open()`, modos y `with`

### La función `open()`

```python
open(file, mode="r", encoding="utf-8")
```

| Parámetro | Descripción |
|---|---|
| `file` | Ruta al archivo (string o `Path`) |
| `mode` | Modo de apertura (ver tabla abajo) |
| `encoding` | Codificación de caracteres. Siempre usar `"utf-8"` para soportar español |

### Modos de apertura

| Modo | Nombre | Crea si no existe | Borra contenido previo | Posición inicial |
|---|---|---|---|---|
| `"r"` | Solo lectura | ❌ Error | ❌ | Inicio |
| `"w"` | Escritura (sobrescribe) | ✅ | ✅ | Inicio |
| `"a"` | Append (agrega al final) | ✅ | ❌ | Final |
| `"x"` | Creación exclusiva | ✅ (falla si existe) | — | Inicio |

> 💡 Para el CRUD usamos principalmente `"r"` (leer) y `"w"` (escribir/reemplazar todo). El modo `"a"` es útil para logs.

### Por qué usar `with` en lugar de `open()` + `close()`

In [ ]:
# ❌ Sin with: si ocurre un error entre open y close, el archivo queda abierto
# file = open("data/test.txt", "w", encoding="utf-8")
# file.write("algo")
# file.close()   ← puede no ejecutarse si hay un error antes

# ✅ Con with: Python garantiza el cierre aunque ocurra un error
try:
    with open("data/test.txt", "w", encoding="utf-8") as file:
        file.write("Hola desde with\n")
except Exception as e:
    print(e)
print("Archivo creado y cerrado correctamente.")
# Cuando termina el bloque with, file.close() se llama automáticamente

---
## Bloque 3 — Archivos TXT

El formato más simple. Útil para logs, notas y texto libre.
**Para datos estructurados (listas, diccionarios) preferimos CSV o JSON.**

### Crear y escribir

In [ ]:
# Crear el archivo si no existe y escribir contenido
with open("data/log.txt", "w", encoding="utf-8") as file:
    file.write("=== Log del sistema ===\n")
    file.write("Usuario creado: Elon (elon@tesla.com)\n")
    file.write("Usuario creado: Jeff (jeff@aws.com)\n")

print("Archivo log.txt creado.")

### Leer el contenido completo vs línea por línea

In [ ]:
# # Leer todo el contenido de una vez
# with open("data/log.txt", "r", encoding="utf-8") as file:
#     contenido = file.read()
#     print("--- read() ---")
#     print(contenido)

# Leer línea por línea (útil para archivos grandes)
with open("data/log.txt", "r", encoding="utf-8") as file:
    print("--- línea por línea ---")
    for linea in file:
        print(linea.lower().strip())   # strip() elimina el \n al final

In [ ]:
# Agregar una línea al final sin borrar el contenido (modo "a")
with open("data/log.txt", "a", encoding="utf-8") as file:
    file.write("Usuario eliminado: id=3\n")

# Verificamos
with open("data/log.txt", "r", encoding="utf-8") as file:
    print(file.read())

### ¿Cuándo usar TXT?

✅ Logs de operaciones del sistema
✅ Archivos de configuración simple
❌ Listas de usuarios, temas, ratings → usar **JSON**

---
## Bloque 4 — Archivos CSV

CSV (Comma Separated Values) es el formato estándar para **datos tabulares**: Excel, Google Sheets, pandas.
Es ideal para **listas de listas**. Para listas de diccionarios, JSON es más natural (ver Bloque 5).

### El módulo `csv`

```python
import csv

csv.writer(file)          # escribe filas como listas
csv.reader(file)          # lee filas como listas
```

### Crear y escribir con headers

Representamos los usuarios como **lista de listas** (cada fila es un usuario):

In [ ]:
import csv

# Dataset: lista de listas
usuarios_tabla = [
    [1, "Elon", "elon@tesla.com"],
    [2, "Jeff", "jeff@aws.com"],
    [3, "Bill", "Bill@microsoft.com"],
]
headers = ["id_usuario", "nombre", "email"]

# Crear el archivo CSV con headers
with open("data/usuarios.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(headers)          # writerow: escribe UNA fila
    writer.writerows(usuarios_tabla)  # writerows: escribe TODAS las filas de una vez

print("usuarios.csv creado.")

### Parámetros importantes de `open()` para CSV

| Parámetro | Valor | Por qué |
|---|---|---|
| `newline=""` | Siempre vacío para CSV | Evita que Python agregue saltos de línea extra en Windows |
| `encoding` | `"utf-8"` | Soporta caracteres especiales |

### Leer un CSV

In [ ]:
import csv

with open("data/usuarios.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    headers = next(reader)    # next() lee la primera fila (headers) y avanza el cursor
    print("Headers:", headers)
    print()
    for fila in reader:
        print(fila)

### ⚠️ CSV solo almacena strings

Al leer un CSV, **todos los valores son strings**: el `id_usuario` que guardamos como `1` vuelve como `'1'`.
Eso rompe cosas como `u["id_usuario"] == id_usuario` en `buscar_usuario_por_id()`. Hay que convertir manualmente:

In [ ]:
import csv

with open("data/usuarios.csv", "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    next(reader)  # saltar headers
    for fila in reader:
        id_usuario = int(fila[0])   # '1' → 1
        nombre     = fila[1]        # string, OK
        email      = fila[2]        # string, OK
        print(f"[{id_usuario}] {nombre} — {email}  (tipo id: {type(id_usuario).__name__})")

### ¿Cuándo usar CSV?

✅ Datos tabulares para exportar a Excel o Google Sheets
✅ Listas de listas con tipos simples (ej: una matriz de ratings)
✅ Integración con pandas
❌ Listas de diccionarios → requiere conversión manual de tipos → usar **JSON**

---
## Bloque 5 — Archivos JSON ⭐

JSON (JavaScript Object Notation) es el formato ideal para **listas de diccionarios**.
Mapea directamente a la lista `usuarios` que hoy tenemos en `datos.py`.

### Ventajas sobre CSV para el proyecto

| Característica | CSV | JSON |
|---|---|---|
| Estructura | Filas y columnas | Objetos anidados |
| Tipos de dato | Todo como string | Preserva int, float, bool, list |
| Conversión al leer | Manual | Automática |
| Legibilidad | Media | Alta |
| Uso en APIs | Poco | Estándar |

### El módulo `json`

```python
import json

json.dump(objeto, file, indent=4, ensure_ascii=False)   # escribe al archivo
json.load(file)                                          # lee del archivo
json.dumps(objeto)                                       # convierte a string
json.loads(string)                                       # parsea desde string
```

### Crear el archivo JSON si no existe

Al igual que "CREATE TABLE IF NOT EXISTS" en SQL, vamos a inicializar el archivo solo si no existe:

In [ ]:
import json
import os

def inicializar_json(ruta):
    """Crea el archivo JSON con una lista vacía si no existe."""
    if os.path.exists(ruta):
        print(f"Archivo ya existe: {ruta}")
        return
    with open(ruta, "w", encoding="utf-8") as file:
        json.dump([], file)
    print(f"Archivo creado: {ruta}")

os.makedirs("data", exist_ok=True)
inicializar_json("data/usuarios.json")

### Escribir: `json.dump()`

Tomamos **exactamente** la lista de `src/datos.py` y la volcamos a disco. Este es el paso que reemplaza a `datos.py`:

In [ ]:
import json

# Contenido de src/datos.py
usuarios = [
    {"id_usuario": 1, "nombre": "Elon", "email": "elon@tesla.com"},
    {"id_usuario": 2, "nombre": "Jeff", "email": "jeff@aws.com"},
    {"id_usuario": 3, "nombre": "Bill", "email": "Bill@microsoft.com"},
    {"id_usuario": 4, "nombre": "Mark", "email": "Mark@meta.com"},
    {"id_usuario": 5, "nombre": "Lisa", "email": "Lisa@amd.com"},
]

with open("data/usuarios.json", "w", encoding="utf-8") as file:
    json.dump(usuarios, file, indent=4, ensure_ascii=False)

print("usuarios.json guardado.")

### Parámetros de `json.dump()`

| Parámetro | Descripción | Recomendación |
|---|---|---|
| `indent` | Espacios de indentación para legibilidad | `4` para lectura humana, `None` para producción |
| `ensure_ascii` | `True`: convierte ñ, á, etc. a `\uXXXX` | `False` para proyectos en español |

### Leer: `json.load()`

In [ ]:
import json

with open("data/usuarios.json", "r", encoding="utf-8") as file:
    usuarios = json.load(file)

print(f"Usuarios cargados: {len(usuarios)}")
print()
for u in usuarios:
    # Los tipos se preservan: id_usuario vuelve como int (¡a diferencia de CSV!)
    print(f"  [{u['id_usuario']}] {u['nombre']} — {u['email']}  (tipo id: {type(u['id_usuario']).__name__})")

### Dos estrategias para combinar memoria y disco

**A. Cargar una vez, guardar al final**

```
Inicio del programa
    └── read_data()  → carga JSON → lista de dicts en memoria
         └── operaciones CRUD (crear, buscar, eliminar...)
              └── write_data()  → lista de dicts → JSON en disco (al salir)
```

Rápida, pero si el programa se corta antes de guardar, se pierden los cambios.

**B. Leer y escribir en cada operación** ← la que usamos en el proyecto

```
crear_usuario()  → read_data() → append en memoria → write_data()
listar_usuarios() → read_data()
```

Un poco más lenta, pero el archivo **siempre** refleja el estado real. Para el tamaño de nuestros datos la diferencia de velocidad es despreciable.

Veamos la estrategia B "a mano", sin funciones auxiliares todavía:

In [ ]:
import json

# 1. Leer el estado actual del disco
with open("data/usuarios.json", "r", encoding="utf-8") as file:
    usuarios = json.load(file)
print(f"Cargados {len(usuarios)} usuarios.")

# 2. Operar en memoria (CREATE)
nuevo = {"id_usuario": 6, "nombre": "Ada", "email": "ada@uade.edu.ar"}
usuarios.append(nuevo)
print(f"Usuario agregado. Total en memoria: {len(usuarios)}")

# 3. Persistir inmediatamente
with open("data/usuarios.json", "w", encoding="utf-8") as file:
    json.dump(usuarios, file, indent=4, ensure_ascii=False)
print("Datos persistidos en disco.")

# 4. Verificar leyendo de nuevo
with open("data/usuarios.json", "r", encoding="utf-8") as file:
    verificacion = json.load(file)
print(f"Verificación: {len(verificacion)} usuarios en disco.")

> 🤔 Fijate que repetimos `with open(...)` + `json.load` / `json.dump` varias veces. Si cada función de `services.py` hiciera esto, tendríamos el mismo código copiado por todos lados. Eso es lo que resuelve el Bloque 6.

---
## Bloque 6 — `read_data()` y `write_data()`: el puente con el CRUD ⭐

Estas dos funciones van en un módulo nuevo, `src/storage.py`, y son el **único** punto de contacto entre los servicios y el disco.
`services.py` las llama para cargar y guardar datos; nunca abre archivos directamente.

```
main.py
  └── services.py
        ├── listar_usuarios()        →  read_data()
        ├── crear_usuario()          →  read_data() + append + write_data()
        └── buscar_usuario_por_id()  →  read_data()
  │
  └── storage.py
        ├── read_data()   →  JSON → lista de dicts
        └── write_data()  →  lista de dicts → JSON
```

Comparado con la versión anterior del proyecto:

| Antes (en memoria) | Ahora (persistente) |
|---|---|
| `from src.datos import usuarios` en `main.py` | `services` lee `data/usuarios.json` |
| `services.crear_usuario(usuarios)` recibe la lista | `services.crear_usuario()` la carga solo |
| `usuarios.append(...)` y listo | `append` + `write_data()` |
| Al cerrar el programa se pierde todo | Los usuarios sobreviven entre ejecuciones |

In [ ]:
import json
import os

# ── src/storage.py ────────────────────────────────────────────────────────────

def read_data(ruta):
    """
    Lee un archivo JSON y retorna su contenido como lista de diccionarios.
    Si el archivo no existe, lo crea vacío y retorna [].

    Parámetros:
        ruta (str | Path): ruta al archivo JSON. Ej: data/usuarios.json

    Retorna:
        list: lista de diccionarios con los registros.
    """
    os.makedirs(os.path.dirname(ruta), exist_ok=True)

    if not os.path.exists(ruta):
        # Comportamiento "CREATE TABLE IF NOT EXISTS":
        # si no existe el archivo, lo creamos vacío y retornamos []
        with open(ruta, "w", encoding="utf-8") as file:
            json.dump([], file)
        return []

    try:
        with open(ruta, "r", encoding="utf-8") as file:
            return json.load(file)
    except (json.JSONDecodeError, OSError) as e:
        print(f"Error al leer {ruta}: {e}")
        return []


def write_data(ruta, datos):
    """
    Escribe una lista de diccionarios en un archivo JSON.
    Sobrescribe el contenido previo (reemplaza todo el archivo).

    Parámetros:
        ruta  (str | Path): ruta al archivo JSON. Ej: data/usuarios.json
        datos (list):       lista de diccionarios a persistir.

    Retorna:
        bool: True si se guardó correctamente, False si hubo error.
    """
    os.makedirs(os.path.dirname(ruta), exist_ok=True)

    try:
        with open(ruta, "w", encoding="utf-8") as file:
            json.dump(datos, file, indent=4, ensure_ascii=False)
        return True
    except OSError as e:
        print(f"Error al escribir {ruta}: {e}")
        return False


# ── Pruebas ───────────────────────────────────────────────────────────────────
RUTA_USUARIOS = "data/usuarios.json"

# read_data: carga el archivo (o lo crea si no existe)
usuarios = read_data(RUTA_USUARIOS)
print(f"read_data() → {len(usuarios)} usuarios cargados.")

# Si el archivo no existiera, read_data() lo crea vacío:
vacio = read_data("data/no_existe_todavia.json")
print(f"read_data() sobre archivo inexistente → {vacio}")

# write_data: persiste todo
ok = write_data(RUTA_USUARIOS, usuarios)
print(f"write_data() → {'OK' if ok else 'ERROR'}")

### Cómo las usa `services.py`

En el proyecto, `crear_usuario()` pide los datos con `interface.get_id_usuario()`, `get_nombre()` y `get_email()` (que usan `input()`).
Para poder correrlo en el notebook, acá usamos una versión que **recibe** el diccionario ya armado: la parte de persistencia es idéntica.

In [ ]:
# ── src/services.py (versión notebook: sin input()) ───────────────────────────

RUTA_USUARIOS = "data/usuarios.json"

def listar_usuarios():
    """READ (todos): levanta el JSON y retorna la lista de usuarios."""
    return read_data(RUTA_USUARIOS)

def buscar_usuario_por_id(id_usuario):
    """READ (uno): retorna el dict del usuario o None."""
    usuarios = read_data(RUTA_USUARIOS)
    return next((u for u in usuarios if u["id_usuario"] == id_usuario), None)

def crear_usuario(usuario):
    """CREATE: agrega un usuario y persiste. False si el id ya existe."""
    usuarios = read_data(RUTA_USUARIOS)
    if any(u["id_usuario"] == usuario["id_usuario"] for u in usuarios):
        print(f"Ya existe un usuario con id {usuario['id_usuario']}.")
        return False
    usuarios.append(usuario)
    return write_data(RUTA_USUARIOS, usuarios)


# ── Demo ──────────────────────────────────────────────────────────────────────
print("=== LISTAR ===")
for u in listar_usuarios():
    print(" ", u)

print()
print("=== CREAR ===")
ok = crear_usuario({"id_usuario": 7, "nombre": "Grace", "email": "grace@navy.mil"})
print(f"crear_usuario(id=7): {'OK' if ok else 'No se creó'}")
ok = crear_usuario({"id_usuario": 7, "nombre": "Otra", "email": "otra@mail.com"})
print(f"crear_usuario(id=7) otra vez: {'OK' if ok else 'No se creó'}")

print()
print("=== BUSCAR ===")
print(f"buscar_usuario_por_id(7): {buscar_usuario_por_id(7)}")
print(f"buscar_usuario_por_id(99): {buscar_usuario_por_id(99)}")

### En el proyecto

La versión real de `src/services.py` agrega la ruta con `pathlib` y los `input()` vía `interface`:

```python
from pathlib import Path
import src.interface as interface
import src.storage as storage

BASE_DIR = Path(__file__).resolve().parent
RUTA_USUARIOS = BASE_DIR.parent / "data" / "usuarios.json"

def listar_usuarios():
    return storage.read_data(RUTA_USUARIOS)

def crear_usuario():
    usuarios = storage.read_data(RUTA_USUARIOS)
    id_usuario = interface.get_id_usuario()
    if any(u["id_usuario"] == id_usuario for u in usuarios):
        print(f"Ya existe un usuario con id {id_usuario}.")
        return False
    nombre = interface.get_nombre()
    email = interface.get_email()
    usuarios.append({"id_usuario": id_usuario, "nombre": nombre, "email": email})
    return storage.write_data(RUTA_USUARIOS, usuarios)
```

Y en `main.py`, la opción "Ver usuarios" pasa a ser:

```python
elif opcion == "2":
    usuarios = services.listar_usuarios()   # levanta el JSON
    interface.mostrar_usuarios(usuarios)
```

👉 Probalo: creá un usuario, cerrá el programa, volvé a abrirlo y elegí "Ver usuarios". Abrí `data/usuarios.json` en VS Code y mirá cómo cambió.

---
## 🏋️ Ejercicios prácticos

### Ejercicio 1 — Completar el CRUD: UPDATE y DELETE

El proyecto hoy tiene CREATE y READ. Usando `read_data()` y `write_data()`, escribí:
- `update_usuario(id_usuario, cambios)` → modifica los campos indicados en `cambios` y persiste. Retorna `True` / `False` si no existe.
- `eliminar_usuario(id_usuario)` → elimina el usuario y persiste. Retorna `True` / `False` si no existe.

> 💡 Para `update`: los diccionarios tienen el método `.update()`.
> 💡 Para `delete`: podés usar `.remove()` sobre la lista o construir una lista nueva sin ese usuario.

Cuando funcione acá, llevalo a `src/services.py` y agregá las opciones 4 y 5 al menú de `main.py`.

In [ ]:
def update_usuario(id_usuario, cambios):
    # Tu solución 👇
    pass

def eliminar_usuario(id_usuario):
    # Tu solución 👇
    pass

# Casos de prueba — no modificar
ok = update_usuario(1, {"email": "elon@x.com"})
print(f"update_usuario(1): {'OK' if ok else 'No encontrado'}")
print(f"Verificación: {buscar_usuario_por_id(1)}")

ok = eliminar_usuario(7)
print(f"eliminar_usuario(7): {'OK' if ok else 'No encontrado'}")
print(f"Verificación: {buscar_usuario_por_id(7)}")

ok = eliminar_usuario(99)
print(f"eliminar_usuario(99): {'OK' if ok else 'No encontrado'}")

### Ejercicio 2 — Log de operaciones en TXT

Agregá una función `log_operacion(operacion, detalle)` que escriba en modo `"a"` en `data/log.txt` cada vez que se realice una operación CRUD.

El formato de cada línea debe ser:
```
[2026-10-26 10:32:01] CREATE | Usuario creado: Grace (grace@navy.mil)
[2026-10-26 10:32:05] DELETE | Usuario eliminado: id=7
```

> 💡 Usá `from datetime import datetime` y `datetime.now().strftime("%Y-%m-%d %H:%M:%S")` para el timestamp.
> 🧩 Extra: ¿en qué módulo del proyecto pondrías esta función? ¿Quién debería llamarla?

In [ ]:
from datetime import datetime

def log_operacion(operacion, detalle):
    # Tu solución 👇
    pass

# Casos de prueba — no modificar
log_operacion("CREATE", "Usuario creado: Grace (grace@navy.mil)")
log_operacion("UPDATE", "Email actualizado: id=1, nuevo email=elon@x.com")
log_operacion("DELETE", "Usuario eliminado: id=7")

# Verificar el contenido del log
with open("data/log.txt", "r", encoding="utf-8") as f:
    print(f.read())

### 🏆 Ejercicio 3 — Desafío: exportar usuarios a CSV

Escribí una función `exportar_usuarios_csv(ruta_json, ruta_csv)` que:
1. Lea los usuarios desde el JSON con `read_data()`
2. Los escriba en un CSV con headers (`id_usuario`, `nombre`, `email`)
3. Retorne la cantidad de registros exportados

Esto simula una funcionalidad real: exportar los usuarios del sistema a Excel para un reporte.

In [ ]:
import csv

def exportar_usuarios_csv(ruta_json, ruta_csv):
    # Tu solución 👇
    pass

# Caso de prueba — no modificar
cantidad = exportar_usuarios_csv("data/usuarios.json", "data/usuarios_export.csv")
print(f"Exportados: {cantidad} usuarios")

# Verificar el CSV
with open("data/usuarios_export.csv", "r", encoding="utf-8") as f:
    print(f.read())

---
## 💬 Reflexión

Respondé en esta celda (doble clic para editar):

**1. ¿Por qué elegirías JSON sobre CSV para persistir la lista de usuarios? ¿Y cuándo usarías CSV?**

> _Tu respuesta acá_

**2. ¿Qué ventaja tiene centralizar la lógica de archivos en `read_data()` y `write_data()` en lugar de usar `open()` directamente en cada función de `services.py`?**

> _Tu respuesta acá_

**3. ¿Qué ocurre si dos personas ejecutan el programa al mismo tiempo y ambas crean un usuario? (Pista: pensá en qué hace `write_data()` con el contenido previo del archivo.)**

> _Tu respuesta acá_